# Datamine COMPSE Process Example

This notebook demonstrates how to configure and run the **`compse`** process wrapper in `dmstudio`.

## Process Description

## COMPSE

Composites drillhole data by optimizing the composite interval using ore and waste criteria.

The input file must be in standard drillhole sample format (as output by the processes [DESURV](<desurv.md>) and [HOLES3D](<holes3d.md>)).

The output file OUT has a similar format as in the input file. The compositing depends on whether a specified numeric field in the input file is above or below a specified cutoff. This file contains samples which have the FROM and TO values of the composited samples. It will generally have fewer samples than the input file. This file is useful to analyse the number of ore and waste composites that have been identified, but it should be noted that if the input file has non-straight holes the curvature may be lost in this file.

The HOLESOUT file contains the same number of records and the same sample geometry as in the input file but has composited grade values. It may also have the optional OWCODE field which flags whether a sample is ore (OWCODE=1) or waste (OWCODE=0). This file is particularly useful when the input file contains non-straight holes. Any curvature is lost in the OUT file which contains just the composited FROM and TO values.

To compare the input grades with composited grades a copy of the VALUE grade field can be made before using **COMPSE** , e.g. by using the [EXTRA](<extra.md>) process.

It is possible that some output composite samples will have a grade value greater than the **CUTOFF** but be flagged as waste due to internal waste or minimum ore width constraints.

Up to 50 explicit numeric fields may be composited at the same time, and written to the OUT file, but they have no influence on which intervals are grouped together. These extra fields do not have to be specified; they are identified by the process as those numeric fields which are not the standard ones. Standard field names are: BHID, X, Y, Z, LENGTH, A0, B0, C0, RADIUS, FROM, TO.

A maximum of 5,000 samples per hole can be composited.

### COMPSE Rules

The following notes describe the rules used by **COMPSE** to determine the ore and waste composites.

Flag all samples as either ore or waste according to cutoff. Samples with a value greater than or equal to **CUTOFF** are flagged as ore. The remaining samples are flagged as waste. (Note if the **MINGLEN** parameter is set then samples with a length*value > **MINGLEN** are also flagged as ore).

##### Rule 1

1. Loop through all samples and find all candidate groups of three samples that can be combined with the following configuration:

1. Wide ore - Narrow waste - Any ore
2. Any ore - Narrow waste - Wide ore
2. Select the candidate group with the highest grade and composite into a new ore sample.
3. Keep executing Rule 1 until no candidate groups remain.
4. If NARWASTE=2 then the three samples are only considered as a candidate if both adjacent ore samples can independently carry the waste. If NARWASTE=1 then the three samples are considered as a candidate as long as either ore can carry the waste.
5. Go to Rule 2.

##### Rule 2

1. Loop through all samples (those composited after rule 1) and look for candidates of 3 samples with the following configuration:
1. If ANYORE2=0 : Narrow ore \- Narrow waste - Narrow ore
2. If ANYORE2=1: Any ore - Narrow waste - Any ore
2. If **NARWASTE** =2 then the three samples are only considered if both adjacent ore samples can independently carry the waste. If **NARWASTE** =1 then the three samples are considered as a candidate as long as either adjacent ore can carry the waste.
3. Select the candidate with the highest grade and composite into a new ore sample.
4. If a candidate sample was found and converted go back to Rule 1.
5. Go to Rule 3.

##### Rule 3

1. If **DILUTE** =0 go to Rule 4.
2. Loop through all samples and see if any narrow ore composites can be combined with adjacent waste samples to create a diluted ore composite. Convert the one with the highest grade.
1. Waste - Narrow ore
2. Narrow ore - Waste
3. Go to Rule 1.

##### Rule 4

1. Change any narrow ore composites to be waste and combine with adjacent waste samples.
2. Go to Rule 5.

##### Rule 5

1. If DILUTE=0 then finish
2. For any narrow waste composites find the adjacent ore with the smallest value of length*grade and change it to waste.
3. Composite the two samples.
4. Repeat until no narrow waste composites are found.
5. Finish

### Notes

* DILUTE=1 will tend to create longer composites of lower grade. It is more aggressive DILUTE=0.
* NARWASTE=1 will tend to create longer composites of lower grade. It is more aggressive than NARWASTE=2
* ANYORE2=1 will tend to combine more samples. It is more aggressive than ANYORE2=0
* DILUTE=1 triggers rules 3 and 5 which are more mining based; they remove narrow samples.
* Rule 4, which converts narrow ore composites to waste is always applied.

### Input Files:

* **in** (Drillhole):
  Input sample file, in BHID and FROM order. This must contain the fields
  BHID,X,Y,Z,FROM,TO and LENGTH
  Required=Yes

### Output Files:

* **out** (Drillhole):
  Output file of ore and waste composites.
  Required=Yes

### Fields:

* **value** (Numeric : IN):
  Numeric value field used to control compositing. This may be a grade or a calculated
  equivalent value from grades of different metals.
  Default=Undefined
  Required=Yes

* **zone** (Numeric : IN):
  Name of optional field for compositing within. This field must exist in the input file
  and can be numeric or alpha. It will be copied to the output file. If specified then new
  composites will be created every time the value of ZONE changes.
  Default=Undefined
  Required=No

* **owcode** (Alphanumeric : -):
  Output field to contain an Ore/Waste Flag. This will contain values of zero for waste
  samples and values of one (1) for Ore samples. It is possible that some output composite
  samples will have a grade value greater than the CUTOFF but be flagged as waste due to
  internal waste or minimum ore width constraints.
  Default=Undefined
  Required=No

### Parameters:

* **cutoff**:
  Minimum value of **VALUE** which is considered to be ore (0).
  Range=Undefined
  Values=Undefined
  Default=0
  Required=Yes

* **minore**:
  Minimum mining width for ore. This must be greater than zero. Output samples must be at
  least this long to be considered as ore, but see also the MINGLEN parameter.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **minglen**:
  Minimum value of sample length multiplied by **VALUE** that a sample must have to be
  considered to be ore. The default value is unset ( or zero ). A value of zero is treated
  as unset. This can be used to treat samples with a length below the minimum mining width
  as ore when they have a relatively high grade.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **maxwaste**:
  Maximum width for internal waste (0.00001). This must be greater than zero.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=Yes

* **minasfr**:
  Minimum fraction of a composited interval which must be assayed for the average assayed
  value to be given to the composite. This only applies to the extra fields mentioned
  above. If **VALUE** is 'absent' in a particular record, its default value is used.
  (0.95)
  Range=0,1
  Values=Undefined
  Default=0.95
  Required=Yes

* **dilute**:
  Attempt to dilute composites to remove samples with a length less than the minum ore or
  minimum waste lengths. =0 : Do not attempt to dilute samples. This is more
  conservative.. =1 : Attempt to dilute narrow ore samples with adjacent waste and narrow
  waste samples with adjacent ore. This is the more aggressive option and will tend to
  create longer ore composites of lower grade.
  Range=
  Values=
  Default=
  Required=No

* **narwaste**:
  Test for carrying narrow waste to be applied to either [1] or both [2] adjacent ores
  (1). =1 : When compositing samples with configuration: ore - Narrow waste - ore, then
  proceed if either adjacent ore sample can carry the internal waste. This is the more
  aggressive option and will tend to create longer ore composites of lower grade.. =2 :
  When compositing samples with configuration: ore - Narrow waste - ore, then proceed only
  if both adjacent ore samples can independently carry the internal waste.
  Range=1,2
  Values=1,2
  Default=1
  Required=Yes

* **anyore2**:
  When applying rule 2 (see start of this Help file) specify whether narrow waste is
  composited with adjacent narrow ore or wide ore. =0 : When applying Rule 2 composite
  samples with configuration: Narrow ore - Narrow waste - Narrow ore. =1 : When applying
  Rule 2 composite samples with configuration: Any ore - Narrow waste - Any ore. This is
  the more aggressive option and will tend to create longer ore composites of lower
  grade..
  Range=0,1
  Values=0,1
  Default=1
  Required=Yes

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('compse')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute compse
print("Running compse...")
dm_cmd.compse(
    in_i='t_assays',  # required
    out_o='t_compse_out',  # required
    value_f='optional',  # required
    cutoff_p='required_val',  # required
    minore_p='required_val',  # required
    maxwaste_p='required_val',  # required
    minasfr_p='required_val',  # required
    narwaste_p='required_val',  # required
    anyore2_p='required_val',  # required
    # zone_f='optional',  # optional
    # owcode_f='optional',  # optional
    # minglen_p='optional',  # optional
    # dilute_p='optional',  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("compse execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_compse_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")